In [1]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Download an example reference data point from LangSmith

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [3]:
ls_client = Client()

### Build Eval Dataset

In [4]:
coordinator_eval_dataset = [
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "What is the weather today?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can I get some earphones?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add an item with ID B09NLTDHQ6 to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "shopping_cart_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add those earphones to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add the best items to my cart? I am looking for laptop bags."}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you find some good reviews for items in my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "shopping_cart_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you put the items with the most positive user reviews to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "What kind of stuff do you sell?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you help me with my order?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "",
                "final_answer": True
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you add two, ideally red tablets to my cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you reserve a red leather laptop bag for me?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "product_qna_agent",
                "final_answer": False
            }
        }
    },
    {
        "inputs": {
            "input": {
                "messages": [
                    {"role": "user", "content": "Can you reserve my shopping cart?"}
                ]
            }
        },
        "outputs": {
            "coordinator_agent": {
                "next_agent": "shopping_cart_agent",
                "final_answer": False
            }
        }
    }
]

### Upload the dataset to LangSmith

In [9]:
dataset_name = "coordinator-first-delegation-evaluation"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="This dataset contains the first delegation of the coordinator agent."
)

In [10]:
dataset

Dataset(name='coordinator-first-delegation-evaluation', description='This dataset contains the first delegation of the coordinator agent.', data_type=<DataType.kv: 'kv'>, id=UUID('a7a10b08-5ee0-45d3-b0a5-ed5d53b07abf'), created_at=datetime.datetime(2026, 8, 18, 21, 44, 40, 923339, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 8, 18, 21, 44, 40, 923339, tzinfo=TzInfo(0)), example_count=None, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'sdk_version': '0.9.3', 'library': 'langsmith', 'platform': 'macOS-15.7.5-x86_64-i386-64bit', 'runtime': 'python', 'py_implementation': 'CPython', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'langchain_core_version': '1.4.8'}})

In [11]:
for item in coordinator_eval_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs=item["inputs"],
        outputs=item["outputs"]
    )

### Coordinator Agent Evaluation

In [12]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

from langchain_core.messages import SystemMessage, AIMessage
from IPython.display import display

from typing import Any, Annotated, List
from pydantic import Field
from operator import add

from jinja2 import Template
from utils.utils import postprocess_response

In [13]:
MAX_ITERATIONS_PRODUCT_QNA = 5
MAX_ITERATIONS_SHOPPING_CART = 4
MAX_ITERATIONS_WAREHOUSE_MANAGER = 4
MAX_ITERATIONS_COORDINATOR = 6

In [14]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [15]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available agents and skills
 
### `product_qna_agent`
 
Owns all product knowledge: catalog, specs, pricing, reviews, recommendations. This is the default agent for any question about what exists or what to buy.
 
Skills:
 
- `get_formatted_item_context` — Search available products and return the top k matching inventory items.
- `get_formatted_reviews_context` — Get the top k reviews matching a query for a list of prefiltered items.
Does not: modify the cart, check warehouse stock, or reserve anything.
 
### `shopping_cart_agent`
 
Owns the state of the user's shopping cart.
 
Skills:
 
- `add_to_shopping_cart` — Add a list of provided items to the shopping cart.
- `get_shopping_cart` — Retrieve all items in a user's shopping cart.
- `remove_from_cart` —  Remove an item completely from the shopping cart.
Does not: recommend products, check warehouse stock, reserve stock, or place orders.
 
### `warehouse_manager_agent`
 
Owns warehouse inventory and reservations.
 
Skills:
 
- `check_warehouse_availability` — Check availability of items across warehouses, including partial fulfillment options.
- `reserve_warehouse_items` — Reserve items from multiple warehouses in a single transaction.
Does not: recommend products, modify the cart.
"""

    template = Template(prompt_template)

    prompt = template.render()

    if state.coordinator_agent.iteration == MAX_ITERATIONS_COORDINATOR:
        tools = [FinalAgentResponse]
    else:
        tools = [Plan, FinalAgentResponse]

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        tools,
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    next_agent = ""
    next_agent_task = ""

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            next_agent_task = response.tool_calls[0].get("args").get("next_agent_task")
            response = AIMessage(content=f"[coordinator_agent decision] Next agent: {next_agent}. Next agent task: {next_agent_task}")
        else:
            postprocessed_response = postprocess_response(response, "FinalAgentResponse")

            final_answer = postprocessed_response.get("final_answer")
            answer = postprocessed_response.get("answer")
            response = postprocessed_response.get("response")

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "next_agent": next_agent,
            "next_agent_task": next_agent_task
        },
        "answer": answer
    }

### Run against LangSmith

In [16]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [17]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-first-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=6,
    num_repetitions=1
)

/Users/tk/Projects/chatbot_poc_v2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'coordinator-delegation-5961ef59' at:
https://smith.langchain.com/o/b99d0ed7-3848-4b7d-a09d-6fa565e05a8b/datasets/a7a10b08-5ee0-45d3-b0a5-ed5d53b07abf/compare?selectedSessions=651e5c00-89cb-494b-ba1b-d88f2a6374de




12it [00:07,  1.61it/s]


### Extract Evaluation Results

In [18]:
results

,inputs.input,outputs.messages,outputs.coordinator_agent,outputs.answer,error,reference.coordinator_agent,feedback.evaluate_coordinator_delegation,execution_time,example_id,id
0,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'shopping_cart_agent', 'final_a...",True,3.433600,3ca3c4b8-9c8b-4109-9366-c41bf083127c,01a016d7-a94f-73a2-b1db-9ae5abbebf0a
1,"{'messages': [{'role': 'user', 'content': 'Wha...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': '', 'final_answer': True}",False,3.430477,efd5e6be-5fda-4ddf-a26d-5a7840d6dd94,01a016d7-a952-74b1-905d-cd28492df0d8
2,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'product_qna_agent', 'final_ans...",True,3.837335,5561a639-6dc3-4531-ab9b-679fd1259fc7,01a016d7-a952-7121-97c3-f959b43424a1
3,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'product_qna_agent', 'final_ans...",True,4.469451,ed06677b-e56b-448c-a180-31aab3d0332c,01a016d7-a950-7bd2-aaa7-0478210fcdb0
4,"{'messages': [{'role': 'user', 'content': 'Can...",[content=': Sure — what would you like help wi...,"{'iteration': 1, 'final_answer': True, 'next_a...",Sure — what would you like help with on your o...,None,"{'next_agent': '', 'final_answer': True}",True,4.521454,94f17aab-e316-42f4-9c27-42e0b1607868,01a016d7-a951-7162-9669-7c12bea74a51
5,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'product_qna_agent', 'final_ans...",True,5.065729,df854a21-08de-43ff-9878-dd27fa31039a,01a016d7-a94f-7d21-9970-49591b783bbe
6,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'shopping_cart_agent', 'final_a...",True,1.563271,3b89ba82-adb8-4b3e-a815-742eb1d6fb2a,01a016d7-bac6-7963-b6f2-a04d9fedca2a
7,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'product_qna_agent', 'final_ans...",True,1.605644,53c11b64-56d7-4528-b799-0468d8b7f9b6,01a016d7-bafb-78a3-9629-efbf1f55ae62
8,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': 'product_qna_agent', 'final_ans...",True,2.746617,7cd3691a-2456-461c-af3d-35adaab58924,01a016d7-b6bb-7100-94af-34bbda836f01
9,"{'messages': [{'role': 'user', 'content': 'Can...",[content='[coordinator_agent decision] Next ag...,"{'iteration': 1, 'final_answer': False, 'next_...",,None,"{'next_agent': '', 'final_answer': True}",False,2.438160,ddadc3a6-9cc2-48ba-8da8-86050bcf34d7,01a016d7-b850-7913-abfa-613878a7fdeb


In [19]:
results.experiment_name

'coordinator-delegation-5961ef59'

In [20]:
results_resp = ls_client.read_project(
    project_name=results.experiment_name,
    include_stats=True
)

In [21]:
results_resp

TracerSessionResult(id=UUID('651e5c00-89cb-494b-ba1b-d88f2a6374de'), start_time=datetime.datetime(2026, 8, 18, 21, 47, 7, 424096, tzinfo=TzInfo(0)), end_time=None, description=None, name='coordinator-delegation-5961ef59', extra={'__progress': {'evaluator_keys': ['evaluate_coordinator_delegation'], 'expected_run_count': 12, 'num_examples': 12, 'num_repetitions': 1}, 'metadata': {'__ls_runner': 'py_sdk_evaluate', 'dataset_splits': ['base'], 'dataset_version': '2026-08-18T21:45:08.028921+00:00', 'git': {'author_email': 'kenner.g.tara@gmail.com', 'author_name': 'tkenner-k', 'branch': 'sprint_6', 'commit': 'c1162d4772584c337bf73740155ee93205bc97a6', 'commit_time': '1786913351', 'dirty': True, 'remote_url': 'https://github.com/tkenner-k/chatbot_poc.git', 'repo_name': 'chatbot_poc_v2', 'tags': None}, 'num_repetitions': 1, 'revision_id': 'c1162d4'}}, tenant_id=UUID('b99d0ed7-3848-4b7d-a09d-6fa565e05a8b'), reference_dataset_id=UUID('a7a10b08-5ee0-45d3-b0a5-ed5d53b07abf'), run_count=12, latency_

In [22]:
results_resp.feedback_stats

{'evaluate_coordinator_delegation': {'n': 8,
  'avg': 0.75,
  'median': None,
  'stdev': 0.4330127018922193,
  'min': 0,
  'max': 1,
  'errors': 0,
  'values': {},
  'type': 'primary',
  'contains_thread_feedback': False}}

In [23]:
results_resp.feedback_stats.get("evaluate_coordinator_delegation")

{'n': 8,
 'avg': 0.75,
 'median': None,
 'stdev': 0.4330127018922193,
 'min': 0,
 'max': 1,
 'errors': 0,
 'values': {},
 'type': 'primary',
 'contains_thread_feedback': False}

In [24]:
results_resp.feedback_stats.get("evaluate_coordinator_delegation").get("avg")

0.75

In [25]:
results_resp.feedback_stats.get("evaluate_coordinator_delegation").get("errors")

0